# Multi-Hop Retrieval — Pipeline Bottleneck Diagnostics

**Before running:**
1. Settings → Accelerator → **T4 GPU**
2. Settings → Internet → **On**
3. Your Google Drive `musique_data/` folder must contain:
   - `musique_ans_v1.0_dev.jsonl`
   - `musique_ans_v1.0_train.jsonl`
   - `model1_complement.pt`
   - `model2_scorer.pt`

**What this does — 5 targeted tests:**
1. **M1 Discriminability** — does `complement(A,B_pos)` point toward C more than `complement(A,B_neg)`?
2. **Edge Coverage** — what % of true hop pairs have a graph edge?
3. **M2 Ranking** — among graph neighbors of A, where does B_pos rank under ColBERT vs cosine?
4. **Agreement** — does ColBERT help or hurt over cosine?
5. **Beam Reach (Oracle Seed)** — given correct 1st hop, does beam find 2nd hop?

Expected time: **~15 min** on T4 (500 val examples, 300 samples per test)

In [ ]:
# ── [EDIT IF NEEDED] ───────────────────────────────────────────────────────────
REPO_URL        = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME       = "multihop-retrieval"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"   # musique_data folder ID
WORK_DIR        = f"/kaggle/working/{REPO_NAME}/retrieval"

MAX_EXAMPLES = 500    # val examples to load (500 = fast but representative)
MAX_SAMPLES  = 300    # samples per diagnostic test
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
# 1. Verify GPU (must be T4, not P100)
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings → Accelerator → T4 GPU")

cc = torch.cuda.get_device_properties(0).major
if cc < 7:
    raise RuntimeError(
        f"GPU is P100 (sm_60) — not supported by PyTorch 2.x.\n"
        "FIX: Settings → Accelerator → change P100 → T4 GPU"
    )

print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("CUDA OK")

In [ ]:
# 2. Clone repo + install dependencies
import os

repo_root = f"/kaggle/working/{REPO_NAME}"
if not os.path.isdir(repo_root):
    os.system(f"git clone {REPO_URL} {repo_root}")
else:
    os.system(f"cd {repo_root} && git pull")

os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())

# Kaggle already has torch + faiss — only install missing packages
os.system("pip install -q 'transformers>=4.35.0' 'rank_bm25>=0.2.2' 'sentence-transformers>=2.2.0' gdown")

try:
    import faiss
    print(f"faiss: OK (GPU={faiss.get_num_gpus()} GPUs)")
except ImportError:
    os.system("pip install -q faiss-cpu")
    print("faiss-cpu installed as fallback")

print("Dependencies ready")

In [ ]:
# 3. Download data + model checkpoints from Google Drive
import os, gdown

download_dir = "/kaggle/working/drive_data"

if not os.path.isdir(download_dir):
    print("Downloading from Google Drive...")
    gdown.download_folder(
        id=DRIVE_FOLDER_ID,
        output=download_dir,
        quiet=False,
        use_cookies=False,
    )
else:
    print("Drive data already downloaded")

print("\nDownloaded files:")
for f in sorted(os.listdir(download_dir)):
    size = os.path.getsize(f"{download_dir}/{f}") / 1e6
    print(f"  {f:45s}  {size:7.1f} MB")

In [ ]:
# 4. Set up expected file paths
import os, shutil

drive = "/kaggle/working/drive_data"

os.makedirs(f"{WORK_DIR}/data/musique", exist_ok=True)
os.makedirs(f"{WORK_DIR}/models", exist_ok=True)
os.makedirs(f"{WORK_DIR}/cache", exist_ok=True)

# MuSiQue JSONL → symlink (avoid copying 260 MB)
for fname in ["musique_ans_v1.0_train.jsonl", "musique_ans_v1.0_dev.jsonl"]:
    src = f"{drive}/{fname}"
    dst = f"{WORK_DIR}/data/musique/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"  {fname}: OK")

# Model checkpoints → copy to models/
for ckpt in ["model1_complement.pt", "model2_scorer.pt"]:
    src = f"{drive}/{ckpt}"
    dst = f"{WORK_DIR}/models/{ckpt}"
    if not os.path.exists(dst):
        print(f"  Copying {ckpt} ({os.path.getsize(src)/1e6:.0f} MB)...", flush=True)
        shutil.copy(src, dst)
    print(f"  {ckpt}: OK ({os.path.getsize(dst)/1e6:.0f} MB)")

print("\nAll paths ready")

---
## Run Diagnostics
Runs 5 tests to find where the pipeline loses points.

In [ ]:
# 5. Run pipeline diagnostics
import os
os.chdir(WORK_DIR)

result_file = "/kaggle/working/diag_results.txt"
cmd = (
    f"python diagnose_models.py "
    f"--max_examples {MAX_EXAMPLES} "
    f"--max_samples {MAX_SAMPLES} "
    f"2>&1 | tee {result_file}"
)
os.system(cmd)
print("\nDiagnostics complete")

In [ ]:
# 6. Display results
with open("/kaggle/working/diag_results.txt") as f:
    print(f.read())